# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata name and description
print("Dataset title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)

# The following displays extra info about the dataset
print("\nPublished:", getattr(dataset.metadata, "datePublished", "N/A"))

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields. All entities are referenced by their Croissant `@id`.

In [ ]:
# List all record sets in the dataset and their fields by @id
record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: @id='{rs.id}', name='{rs.name}'")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - @id='{field.id}', name='{field.name}', dataType='{field.data_type}'")
        print()

# For the rest of the notebook, we will use the @id values from here

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s found above.
If there are no record sets, the notebook will not proceed further,

In [ ]:
# Identify the record set(s) from the metadata
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets] if record_sets else []

dataframes = {}

for rs in record_sets:
    print(f"Loading records from RecordSet @id='{rs.id}' ...")
    # Each record is a dict with field @id as the key
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"- Shape: {df.shape}")
    print(f"- Columns (fields): {list(df.columns)}\n")

if dataframes:
    # Just use the first record set for examples below
    example_record_set_id = record_set_ids[0]
    print(f"First 5 records from record set: {example_record_set_id}")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, or grouping. 
Reference fields by their `@id`.

Below, we select a numeric field. Use the list of fields from the overview to choose an appropriate field.

In [ ]:
if not dataframes:
    print("No record sets with data available. Skipping EDA.")
else:
    df = dataframes[example_record_set_id]
    # Identify candidate numeric fields by their @id (from step 2). Here we auto-detect numeric columns.
    # Example: numeric_field_id = 'cr:abundance' (replace with your actual field @id)
    numeric_field_id = None
    for col in df.columns:
        try:
            # Convert first value to float to test
            float(df[col].dropna().iloc[0])
            numeric_field_id = col
            break
        except:
            continue
    if not numeric_field_id:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field '@id'='{numeric_field_id}' for analysis.")

        # Convert column to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
            / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Example group field: pick first non-numeric
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Visualize the numeric field and grouped statistics if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not numeric_field_id:
    print("No data available for visualization.")
else:
    fig, ax = plt.subplots(1,2, figsize=(12,5))

    # Histogram of numeric field
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax[0])
    ax[0].set_title(f"Histogram of {numeric_field_id}")

    # If group_field_id and grouped_df available, plot bar plot
    if 'grouped_df' in locals() and group_field_id:
        grouped_df.plot(kind='bar', legend=False, ax=ax[1])
        ax[1].set_title(f"Mean {numeric_field_id} grouped by {group_field_id}")
    else:
        ax[1].axis('off')

    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated programmatic exploration of a Croissant-FAIR dataset using `mlcroissant`. 

- We loaded the metadata, listed available record sets and fields by `@id`,
- Extracted data using the `records` API,
- Performed simple EDA and normalization on a numeric field,
- Visualized distributions and group means.

For a full analysis, integrate domain knowledge about the fields (identified by `@id`) and tailor analytics to specific hypotheses.